# Cross-validation, metrics, and model selection

This notebook practices **train/validation** thinking with `Pipeline`, `cross_val_score`, and `GridSearchCV` on a small classification task.

In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd().resolve()
for _ in range(12):
    if (_repo / "utils" / "data_paths.py").exists():
        sys.path.insert(0, str(_repo))
        break
    if _repo.parent == _repo:
        raise FileNotFoundError("Run from ml-notebook root.")
    _repo = _repo.parent

from utils.data_paths import data_dir
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv(data_dir() / "Bank_Personal_Loan_Modelling.csv")
X = df.drop(columns=["ID", "Personal Loan"])
y = df["Personal Loan"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
print("ROC-AUC folds:", scores.round(4))
print("Mean:", scores.mean().round(4), "+/-", scores.std().round(4))

pipe.fit(X_train, y_train)
y_score = pipe.predict_proba(X_test)[:, 1]
print("Test ROC-AUC:", roc_auc_score(y_test, y_score).round(4))
print(classification_report(y_test, pipe.predict(X_test), digits=3))

## Try this next

- Swap `scoring` to `f1`, `average_precision`, or `neg_log_loss` and interpret tradeoffs.
- Replace `LogisticRegression` with `RandomForestClassifier` inside the same `Pipeline` (drop `StandardScaler` for tree models).
- Add `GridSearchCV` over `C` for logistic regression.